# DeepSets vs GraphSAGE Comparison

Compare DeepSets (coordinate-based) with GraphSAGE for surface code decoding.

**Metrics:**
- Accuracy at each distance
- Logical Error Rate (LER)
- Inference speed
- Parameter count

**Distances:** d=3, 5, 7, 9 (extrapolation test)

In [ ]:
import sys
import json
import time
from pathlib import Path
from datetime import datetime

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/code')
else:
    BASE_PATH = Path('..').resolve().parent  # code/deepsets/results -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from benchmark_models import DeepSets, DeepSetsModel, SurfaceCodeSampler

# Paths
RESULTS_DIR = BASE_PATH / "results" / "deepsets_vs_gnn_comparison"
PLOTS_DIR = RESULTS_DIR / "plots"
OUTPUT_DIR = RESULTS_DIR / "results"
BASELINE_RESULTS_PATH = BASE_PATH / "results" / "nn_vs_gnn_comparison" / "results" / "comparison_results.json"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Configuration

In [ ]:
# Distances to evaluate
DISTANCES = [3, 5, 7, 9]

# Test samples per distance
TEST_SAMPLES = 10000

# Number of benchmark runs for timing
NUM_TIMING_RUNS = 5

# Error rates for testing
P_VALUES = [0.001, 0.002, 0.005, 0.007, 0.01]

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Configuration:")
print(f"  Distances: {DISTANCES}")
print(f"  Test samples per distance: {TEST_SAMPLES:,}")
print(f"  Error rates: {P_VALUES}")

## Load Models

In [ ]:
# Load best DeepSets model from tuning
BEST_CONFIG_PATH = BASE_PATH / "deepsets" / "tuning" / "best_model_config.json"

if BEST_CONFIG_PATH.exists():
    with open(BEST_CONFIG_PATH, 'r') as f:
        best_config = json.load(f)
    print(f"Loaded best config: {best_config['config_id']}")
else:
    print("No best config found, using defaults")
    best_config = {
        'model_params': {
            'phi_hidden': (128, 128),
            'rho_hidden': (256, 128),
            'pool': 'sum',
            'dropout': 0.0,
        }
    }

# Initialize DeepSets model
deepsets = DeepSets(
    nickname="evaluation",
    phi_hidden=tuple(best_config['model_params']['phi_hidden']),
    rho_hidden=tuple(best_config['model_params']['rho_hidden']),
    pool=best_config['model_params']['pool'],
    dropout=0.0,
    device=device,
    base_path=BASE_PATH,
)

# Try to load trained weights
model_path = BASE_PATH / "deepsets" / "extrapolation" / "models" / "d7_only.pt"
if model_path.exists():
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    deepsets.model.load_state_dict(checkpoint['state_dict'])
    print(f"Loaded weights from: {model_path.name}")
else:
    print(f"No trained model found at {model_path}")

## Generate Test Data

In [ ]:
# Generate test data for each distance
test_data = {}

for d in DISTANCES:
    print(f"\nGenerating test data for d={d}...")
    sampler = SurfaceCodeSampler(p=P_VALUES[0], device=device)
    detections, labels = sampler.sample(
        d=d,
        num_samples=TEST_SAMPLES,
        p_values=P_VALUES,
        p_weights=[1.0/len(P_VALUES)] * len(P_VALUES)
    )
    test_data[d] = {'detections': detections, 'labels': labels}
    print(f"  Generated {len(labels):,} samples, shape: {detections.shape}")

## Evaluate Accuracy

In [ ]:
def evaluate_accuracy(model, detections, labels, d, batch_size=512):
    """Evaluate model accuracy."""
    model.model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for i in range(0, len(labels), batch_size):
            batch_det = detections[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            
            preds = model.predict(batch_det, d)
            correct += ((preds > 0.5).float() == batch_labels).sum().item()
            total += len(batch_labels)
    
    return correct / total


# Evaluate on all distances
results = {}

print("\nEvaluating DeepSets on all distances...")
print("-" * 50)

for d in DISTANCES:
    data = test_data[d]
    acc = evaluate_accuracy(deepsets, data['detections'], data['labels'], d)
    ler = 1.0 - acc
    results[d] = {'accuracy': acc, 'ler': ler}
    print(f"d={d}: Accuracy={acc*100:.2f}%, LER={ler*100:.2f}%")

## Inference Speed Benchmark

In [ ]:
# Benchmark inference speed
print("\nBenchmarking inference speed...")

timing_results = {}

for d in DISTANCES:
    data = test_data[d]
    batch = data['detections'][:1000]  # Use 1000 samples for timing
    
    # Warmup
    for _ in range(3):
        _ = deepsets.predict(batch, d)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    times = []
    for _ in range(NUM_TIMING_RUNS):
        start = time.perf_counter()
        _ = deepsets.predict(batch, d)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    
    avg_time = np.mean(times)
    samples_per_sec = 1000 / avg_time
    timing_results[d] = {
        'avg_time_ms': avg_time * 1000,
        'samples_per_sec': samples_per_sec
    }
    print(f"d={d}: {avg_time*1000:.2f}ms / 1000 samples ({samples_per_sec:.0f} samples/sec)")

## Comparison with Baselines

In [ ]:
# Load baseline results
if BASELINE_RESULTS_PATH.exists():
    with open(BASELINE_RESULTS_PATH, 'r') as f:
        baseline_data = json.load(f)
    print("Loaded baseline results.")
else:
    print("WARNING: Baseline results not found.")
    baseline_data = None

# Prepare plot data
plot_distances = sorted(list(set(DISTANCES) & set([int(k) for k in baseline_data['accuracy_results'].keys()])))

# Extract accuracies
deepsets_acc = [results[d]['accuracy'] * 100 for d in plot_distances]
simplenn_acc = [baseline_data['accuracy_results'][str(d)]['simplenn']['accuracy_mean'] * 100 for d in plot_distances]
graphsage_acc = [baseline_data['accuracy_results'][str(d)]['graphsage']['accuracy_mean'] * 100 for d in plot_distances]

# Extract LERs
deepsets_ler = [results[d]['ler'] * 100 for d in plot_distances]
simplenn_ler = [(1 - baseline_data['accuracy_results'][str(d)]['simplenn']['accuracy_mean']) * 100 for d in plot_distances]
graphsage_ler = [(1 - baseline_data['accuracy_results'][str(d)]['graphsage']['accuracy_mean']) * 100 for d in plot_distances]

# Parameter counts
deepsets_params = sum(p.numel() for p in deepsets.model.parameters())
# Get baseline params for d=7 (example)
simplenn_params = next(item['simplenn_params'] for item in baseline_data['parameter_comparison'] if item['distance'] == 7)
graphsage_params = next(item['graphsage_params'] for item in baseline_data['parameter_comparison'] if item['distance'] == 7)

## Visualization

In [ ]:
# Plot 1: Accuracy Comparison
plt.figure(figsize=(10, 6))

plt.plot(plot_distances, deepsets_acc, 'o-', linewidth=2, label='DeepSets (Ours)', color='#2ecc71')
plt.plot(plot_distances, simplenn_acc, 's--', linewidth=2, label='SimpleNN', color='#e74c3c')
plt.plot(plot_distances, graphsage_acc, '^-.', linewidth=2, label='GraphSAGE', color='#9b59b6')

plt.xlabel('Code Distance', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Decoding Accuracy: DeepSets vs Baselines', fontsize=14, fontweight='bold')
plt.xticks(plot_distances)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'comparison_accuracy.png', dpi=150)
plt.show()

In [ ]:
# Plot 2: LER Comparison (Log Scale)
plt.figure(figsize=(10, 6))

plt.semilogy(plot_distances, deepsets_ler, 'o-', linewidth=2, label='DeepSets (Ours)', color='#2ecc71')
plt.semilogy(plot_distances, simplenn_ler, 's--', linewidth=2, label='SimpleNN', color='#e74c3c')
plt.semilogy(plot_distances, graphsage_ler, '^-.', linewidth=2, label='GraphSAGE', color='#9b59b6')

plt.xlabel('Code Distance', fontsize=12)
plt.ylabel('Logical Error Rate (%)', fontsize=12)
plt.title('Logical Error Rate Comparison', fontsize=14, fontweight='bold')
plt.xticks(plot_distances)
plt.grid(True, alpha=0.3, which='both')
plt.legend(fontsize=12)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'comparison_ler.png', dpi=150)
plt.show()

In [ ]:
# Plot 3: Parameter Efficiency (at d=7)
plt.figure(figsize=(8, 5))

param_counts = [deepsets_params, simplenn_params, graphsage_params]
labels = ['DeepSets', 'SimpleNN (d=7)', 'GraphSAGE']
colors = ['#2ecc71', '#e74c3c', '#9b59b6']

bars = plt.bar(labels, param_counts, color=colors, edgecolor='black')
plt.ylabel('Number of Parameters', fontsize=12)
plt.title('Model Parameter Efficiency (d=7)', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.yscale('log')  # Log scale due to large differences

for bar, count in zip(bars, param_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             f'{count/1e3:.1f}k', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'comparison_params.png', dpi=150)
plt.show()

## Summary Report

In [ ]:
print("\n===========================")
print("COMPARISON SUMMARY")
print("===========================")
print(f"{'Distance':<10} {'DeepSets':<12} {'GraphSAGE':<12} {'SimpleNN':<12}")
print("-"*50)

for i, d in enumerate(plot_distances):
    print(f"{d:<10} {deepsets_acc[i]:<12.2f} {graphsage_acc[i]:<12.2f} {simplenn_acc[i]:<12.2f}")

print(f"\nDeepSets Params: {deepsets_params:,}")
print(f"GraphSAGE Params: {graphsage_params:,}")
print(f"SimpleNN Params (d=7): {simplenn_params:,}")